# Vergleich der 3 Prompting-Ansätze (Allgemeine Verbandsklassen)

Dieses Notebook vergleicht die Evaluierungsergebnisse aller 3 Prompt-Ansätze für den generischen Produktkatalog:
1. **Zero-Shot**
2. **Few-Shot** (2 Beispielwunden: `wunde_18` & `wunde_28`)
3. **2-Stage Chain-of-Thought (CoT)**

Für jeden Ansatz wird die Haupt-Metrik (Score / F1-Score (Mean)) und die Genauigkeit (Exact-Match-Rate) einmal **roh (Phase 0)** und einmal **normalisiert (Phase 1)** verglichen sowie der Direktvergleich aller 3 Ansätze, eine Gesamt-Set-Analyse und eine Wortanzahl-Effizienzanalyse dargestellt.

In [ ]:
import sys
import os
import pandas as pd

# Pfad anpassen, um utils_notebook und das eval-Modul zu importieren
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from utils_notebook.metrics_explorer import calculate_scores, calculate_summary
from utils_notebook.plot_compare import plot_gt_comparison
from utils_notebook.prompt_compare import (
    plot_all_prompts_comparison, 
    plot_product_prompts_comparison,
    plot_combined_product_set_comparison,
    display_combined_set_table,
    plot_word_count_analysis,
    display_word_count_table
)

# Gemeinsamer Pfad zur Ground-Truth-Datei
CSV_PATH = "../data/ground_truth/allgemeine_verbandsklassen.csv"

## 1. Zero-Shot Ansatz

Analyse des Zero-Shot-Ansatzes ohne zusätzliche Beispiele.

In [ ]:
JSON_DIR_ZERO_SHOT = "../runs/gpt-5/zero_shot"

# Berechne Roh- und Normalisierte Scores
df_scores_zs_raw = calculate_scores(CSV_PATH, JSON_DIR_ZERO_SHOT, raw=True)
sum_zs_raw = calculate_summary(df_scores_zs_raw)

df_scores_zs_norm = calculate_scores(CSV_PATH, JSON_DIR_ZERO_SHOT, raw=False)
sum_zs_norm = calculate_summary(df_scores_zs_norm)

# Plot: Roh (Phase 0) vs Normalisiert (Phase 1)
plot_gt_comparison(sum_zs_raw, sum_zs_norm, expert_label="Zero-Shot (Allgemeine Verbandsklassen)")

## 2. Few-Shot Ansatz

Analyse des Few-Shot-Ansatzes mit 2 vordefinierten Beispielwunden (`wunde_18` & `wunde_28`).

In [ ]:
JSON_DIR_FEW_SHOT = "../runs/gpt-5/few_shot"

# Berechne Roh- und Normalisierte Scores
df_scores_fs_raw = calculate_scores(CSV_PATH, JSON_DIR_FEW_SHOT, raw=True)
sum_fs_raw = calculate_summary(df_scores_fs_raw)

df_scores_fs_norm = calculate_scores(CSV_PATH, JSON_DIR_FEW_SHOT, raw=False)
sum_fs_norm = calculate_summary(df_scores_fs_norm)

# Plot: Roh (Phase 0) vs Normalisiert (Phase 1)
plot_gt_comparison(sum_fs_raw, sum_fs_norm, expert_label="Few-Shot (Allgemeine Verbandsklassen)")

## 3. 2-Stage Chain-of-Thought (CoT) Ansatz

Analyse des 2-Stage CoT-Ansatzes (Stufe 1: Befundung -> Stufe 2: Produktempfehlung).

In [ ]:
JSON_DIR_TWO_STAGE = "../runs/gpt-5/two_stage"

# Berechne Roh- und Normalisierte Scores
df_scores_2s_raw = calculate_scores(CSV_PATH, JSON_DIR_TWO_STAGE, raw=True)
sum_2s_raw = calculate_summary(df_scores_2s_raw)

df_scores_2s_norm = calculate_scores(CSV_PATH, JSON_DIR_TWO_STAGE, raw=False)
sum_2s_norm = calculate_summary(df_scores_2s_norm)

# Plot: Roh (Phase 0) vs Normalisiert (Phase 1)
plot_gt_comparison(sum_2s_raw, sum_2s_norm, expert_label="2-Stage CoT (Allgemeine Verbandsklassen)")

## 4. Direktvergleich aller 3 Ansätze über alle Kategorien

Gruppiertes Balkendiagramm, das die normalisierten F1-Scores / Mean Scores aller 17 Kategorien für Zero-Shot, Few-Shot und 2-Stage CoT nebeneinander vergleicht.

In [ ]:
# Direktvergleich aller 3 Ansätze für alle 17 Kategorien
plot_all_prompts_comparison(sum_zs_norm, sum_fs_norm, sum_2s_norm)

## 5. Fokus-Vergleich: Produktempfehlungen (Primär- & Sekundärverband)

Gezielter Direktvergleich der 3 Ansätze ausschließlich für die Kernkategorien der Produktempfehlung (`primaerverband` und `sekundaerverband`).

In [ ]:
# Fokus-Vergleich der Produktempfehlungen (Primär- & Sekundärverband)
plot_product_prompts_comparison(sum_zs_norm, sum_fs_norm, sum_2s_norm)

## 6. Analyse des Gesamt-Produktsets pro Wundbild (Primär + Sekundär + Hautschutz)

Hier werden alle empfohlenen Produkte aus **Primärverband (Präferenz & Alternative)**, **Sekundärverband** und **Wundrand/Hautschutz** pro Wundbild in ein einziges Gesamt-Set zusammengefügt und bewertet.

In [ ]:
# Gesamt-Set Plot (Primär- + Sekundärverband + Hautschutz in 1 Set)
plot_combined_product_set_comparison(
    csv_path=CSV_PATH,
    json_dir_zs=JSON_DIR_ZERO_SHOT,
    json_dir_fs=JSON_DIR_FEW_SHOT,
    json_dir_2s=JSON_DIR_TWO_STAGE
)

In [ ]:
# Übersichts-Tabelle des Gesamt-Produktsets pro Wundbild (Zero-Shot)
display_combined_set_table(
    csv_path=CSV_PATH,
    json_dir=JSON_DIR_ZERO_SHOT,
    approach_name="Zero-Shot"
)

## 7. Wortanzahl & Token-Effizienz der Roh-Antworten

Analyse der Textlänge (Wortanzahl) der Roh-Antworten pro Wundbild und Vergleich der Token-Effizienz (Gesamt-Set F1-Score % / Ø Wortanzahl).

In [ ]:
# Plot: Ø Wortanzahl pro Wundbild und Token-Effizienz Index
plot_word_count_analysis(
    json_dir_zs=JSON_DIR_ZERO_SHOT,
    json_dir_fs=JSON_DIR_FEW_SHOT,
    json_dir_2s=JSON_DIR_TWO_STAGE
)

In [ ]:
# Detailtabelle: Wortanzahl der Roh-Antworten für jedes Wundbild über alle 3 Ansätze
display_word_count_table(
    json_dir_zs=JSON_DIR_ZERO_SHOT,
    json_dir_fs=JSON_DIR_FEW_SHOT,
    json_dir_2s=JSON_DIR_TWO_STAGE
)

## 8. Zusammenfassung der Ergebnisse für die Masterarbeit

### 📌 1. Gesamt-Performance über alle 17 Kern-Kategorien (Normalisiert Phase 1)
* **Evaluierte Kategorien:** 17 klinische Parameter und Produktklassen (L&R-Spezialfelder wie `wundgrund` wurden herausgefiltert).
* **Ergebnisse (Durchschnittlicher F1-Score):**
  * 🥇 **Zero-Shot:** **57,4 %**
  * 🥈 **Few-Shot:** **55,8 %** *(mit `wunde_18` & `wunde_28` als Beispiele)*
  * 🥉 **2-Stage CoT:** **55,6 %**
* **Erkenntnis:** Alle 3 Ansätze bewegen sich auf einem sehr ähnlichen Niveau (~56 % – 57 %). Zero-Shot schneidet leicht am besten ab, da es keinen *Example-Bias* (Verzerrung durch Beispielsätze im Prompt) entwickelt und keine Information-Loss-Stufen wie der 2-Stage-Ansatz besitzt.

---

### 📌 2. Auswertung der Produktempfehlungen (Primärverband)
* **Primärverband (Best-Path F1):**
  * **Zero-Shot:** **60,4 %** (Exact Match: 21,7 %)
  * **Few-Shot:** **58,4 %** (Exact Match: 15,5 %)
  * **2-Stage CoT:** **56,9 %** (Exact Match: 16,7 %)
* **Ursache für die ~60 % Grenze:** In der Wundversorgung existieren bei fast jeder Wunde mehrere medizinisch gleichwertige Behandlungsoptionen (z. B. Hydrofaser vs. Alginat). Die starre Fixierung der Evaluation auf die Präferenz des einen Experten im Ground Truth bildet eine natürliche Obergrenze.

---

### 📌 3. Gesamt-Produktset Evaluation (Primär + Sekundär + Hautschutz kombiniert)
* **Hintergrund:** Experten nutzen Produkte wie Schaumstoffverbände klinisch flexibel – sowohl als Primärverband als auch als sekundäre Schutzabdeckung. Das Gesamt-Set hebt starre Kategoriegrenzen auf.
* **Ergebnisse (Phase 1 Normalisiert):**
  * 🥇 **2-Stage CoT:** **51,6 %** *(Phase 0 Roh: 40,6 % -> **+11,0 %** durch Mapping)*
  * 🥈 **Zero-Shot:** **49,7 %** *(Phase 0 Roh: 40,6 % -> **+9,1 %** durch Mapping)*
  * 🥉 **Few-Shot:** **47,8 %** *(Phase 0 Roh: 39,7 % -> **+8,1 %** durch Mapping)*
* **Kernerkenntnis:** 
  1. **2-Stage CoT wird beim Gesamt-Set zum Testsieger (51,6 %)**, da die zweistufige Befundung in Stufe 1 zu einer umfassenderen Erfassung aller benötigten Produktkomponenten in Stufe 2 führt.
  2. Das Gesamt-Set übertrifft den Durchschnitt der 3 isolierten Kategorien (**38,5 %**) um **+13,1 Prozentpunkte**, da Zuordnungs-Verwechslungen zwischen Primär- und Sekundärverband aufgefangen werden.

---

### 📌 4. Textlänge (Wortanzahl) & Token-Effizienz
* **Durchschnittliche Wortanzahl der Roh-Antworten pro Wundbild:**
  * **Zero-Shot:** Ø **132,0 Wörter** *(Kompaktestes JSON)*
  * **Few-Shot:** Ø **141,3 Wörter** *(Mittelfeld durch Nachahmung der Prompt-Beispiele)*
  * **2-Stage CoT:** Ø **179,7 Wörter** *(Längster Text durch 2 aufeinanderfolgende Stufen)*
* **Token-Effizienz-Index (Gesamt-Set F1-Score % / Ø Wortanzahl):**
  * 🥇 **Zero-Shot:** **0,377** *(Höchste Effizienz: Sehr hoher F1-Score bei minimalem Token-Verbrauch)*
  * 🥈 **Few-Shot:** **0,338**
  * 🥉 **2-Stage CoT:** **0,287** *(Erreicht zwar die höchste Genauigkeit, benötigt dafür aber 36 % mehr Text)*

---

### 📌 5. Fazit & Empfehlungen

| Anwendungsfall | Empfohlener Prompting-Ansatz | Begründung |
| :--- | :--- | :--- |
| **Beste Gesamt-Abdeckung aller Produkte** | **2-Stage CoT** | Erreicht beim Gesamt-Produktset den besten F1-Score (**51,6 %**). |
| **Höchste Effizienz & Praxistauglichkeit** | **Zero-Shot** | Liefert bei minimaler Wortanzahl (132 Wörter) die höchste Token-Effizienz (**0,377**) und die beste Primärverbands-Quote (**60,4 %**). |